# TrashScan — Path A simplificado

Este notebook assume que a estrutura **já está baixada/criada no volume** do RunPod:

- `/workspace/.venv`
- `/workspace/TrashScan`
- `/workspace/TACO`
- `/workspace/external_datasets`
- `/workspace/processed_4cls`
- `/workspace/runs`
- `/workspace/results_path_A`

O foco aqui é rodar o fluxo do **Path A** usando os scripts `.py` do projeto, sem refazer downloads e sem refazer merge/preprocessamento por padrão.

> Observação: os pesos dos modelos **`yolov11m`**, **`yolov8m`** e **`yolov9s`** já foram calculados e devem estar disponíveis em `runs/path_A`.


## 0) Antes de abrir/rodar o notebook

No terminal do pod, ative a venv do volume:

```bash
cd /workspace
source .venv/bin/activate
```

Se precisar reinstalar dependências:

```bash
cd /workspace/TrashScan/env
python -m pip install -r environment.txt
```

No VSCode/Jupyter, selecione o kernel correspondente a:

```bash
/workspace/.venv/bin/python
```

Para confirmar dentro do notebook, rode a célula abaixo e confira `sys.executable`.


## 1) Imports, paths e utilitários

In [1]:
from pathlib import Path
import os
import sys
import shlex
import subprocess
import shutil

# Caminhos principais no RunPod
WORKSPACE = Path('/workspace')
REPO_ROOT = WORKSPACE / 'TrashScan'

DATA_DIR = REPO_ROOT / 'data'
TRAIN_DIR = REPO_ROOT / 'train' / 'paths'
EVAL_DIR = REPO_ROOT / 'eval'

EXTERNAL_DIR = WORKSPACE / 'external_datasets'
TACO_DIR = WORKSPACE / 'TACO'
PROCESSED_DIR = WORKSPACE / 'processed_5cls'
DATASET_YAML_PATH_A = PROCESSED_DIR / 'dataset_path_A.yaml'

# Treino fica no disco local do pod para evitar I/O lento e cache pesado no volume.
# Se o pod for deletado, salve o essencial de volta no volume antes.
RUNS_PATH_A_DIR = WORKSPACE / 'runs' / 'path_A_5cls'
MLFLOW_DIR = Path('/root/mlflow')

# Resultados finais persistentes no volume
RESULTS_PATH_A_DIR = WORKSPACE / 'results_path_A_5cls'

for p in [RUNS_PATH_A_DIR, MLFLOW_DIR, RESULTS_PATH_A_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python             =', sys.executable)
print('REPO_ROOT          =', REPO_ROOT)
print('TACO_DIR           =', TACO_DIR)
print('EXTERNAL_DIR       =', EXTERNAL_DIR)
print('PROCESSED_DIR      =', PROCESSED_DIR)
print('DATASET_YAML_PATH_A=', DATASET_YAML_PATH_A)
print('RUNS_PATH_A_DIR    =', RUNS_PATH_A_DIR)
print('MLFLOW_DIR         =', MLFLOW_DIR)
print('RESULTS_PATH_A_DIR =', RESULTS_PATH_A_DIR)


def run_cmd(cmd, cwd=WORKSPACE, env=None):
    """Roda comandos de forma previsível no notebook."""
    if isinstance(cmd, str):
        print('$', cmd)
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)

    print('$', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)


Python             = /workspace/.venv/bin/python
REPO_ROOT          = /workspace/TrashScan
TACO_DIR           = /workspace/TACO
EXTERNAL_DIR       = /workspace/external_datasets
PROCESSED_DIR      = /workspace/processed_5cls
DATASET_YAML_PATH_A= /workspace/processed_5cls/dataset_path_A.yaml
RUNS_PATH_A_DIR    = /workspace/runs/path_A_5cls
MLFLOW_DIR         = /root/mlflow
RESULTS_PATH_A_DIR = /workspace/results_path_A_5cls


## 2) Verificação dos arquivos principais

In [2]:
required_paths = [
    DATA_DIR / 'merge_datasets.py',
    DATA_DIR / 'preprocess.py',
    TRAIN_DIR / 'train_path_A.py',
    EVAL_DIR / 'evaluate.py',
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print('Arquivos ausentes:')
    for m in missing:
        print(' -', m)
    raise FileNotFoundError('Há scripts ausentes no clone. Veja a lista acima.')

print('Todos os scripts principais foram encontrados.')
print('Dataset YAML existe?', DATASET_YAML_PATH_A.exists())


Todos os scripts principais foram encontrados.
Dataset YAML existe? True


## 3) Configuração de GPU e treino

In [4]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_properties(0).name
    DEVICE = '0'
else:
    gpu_name = 'cpu'
    DEVICE = 'cpu'

# Ajuste se necessário
EPOCHS = 300
BATCH = 16
IMGSZ = 640
PATIENCE = 15

# Pesos já calculados anteriormente: yolov11m, yolov8m, yolov9s.
# Se rodar a célula de treino abaixo, esses modelos serão treinados novamente.
MODELS = [
    'yolov11m'
]

print('Dispositivo:', gpu_name)
print('DEVICE     :', DEVICE)
print('BATCH      :', BATCH)
print('EPOCHS     :', EPOCHS)
print('IMGSZ      :', IMGSZ)
print('PATIENCE   :', PATIENCE)
print('MODELS     :', MODELS)


Dispositivo: NVIDIA RTX A4500
DEVICE     : 0
BATCH      : 16
EPOCHS     : 300
IMGSZ      : 640
PATIENCE   : 15
MODELS     : ['yolov11m']


## 4) Merge dos datasets — normalmente NÃO rodar

A estrutura já existe em `/workspace/processed_4cls`, então esta célula fica comentada por padrão.

Rode apenas se você mudar os datasets de origem ou precisar reconstruir `merged_data`.


In [5]:
merge_script = DATA_DIR / 'merge_datasets.py'

run_cmd([
    sys.executable, str(merge_script),
    '--taco_root', str(TACO_DIR),
    '--external_root', str(EXTERNAL_DIR),
    '--output_root', str(PROCESSED_DIR),
    '--skip_preprocess',
])


$ /workspace/.venv/bin/python /workspace/TrashScan/data/merge_datasets.py --taco_root /workspace/TACO --external_root /workspace/external_datasets --output_root /workspace/processed_5cls --skip_preprocess


  TACO           : 1500 images, 4784 annotations


  TACO-dataset-1__annotations.coco:   0%|          | 0/1525 [00:00<?, ?it/s]         

  TACO-dataset-1/_annotations.coco: 189 imgs, 346 anns


  TACO-dataset-1__annotations.coco:   2%|▏         | 4/189 [00:00<00:06, 28.56it/s]    

...

  TACO-dataset-3/_annotations.coco: 461 imgs, 884 anns


  TACO-dataset-3__annotations.coco:   0%|          | 0/326 [00:00<?, ?it/s]            

  TACO-dataset-3/_annotations.coco: 2950 imgs, 6615 anns


  TACO-dataset-3/_annotations.coco: 326 imgs, 608 anns

Merged → /workspace/processed_5cls/merged_data/annotations.json
  Total images     : 9519
  Total annotations: 22362
  plastic     : 8765
  paper       : 1860
  metal       : 1793
  glass       : 695
  other       : 9249
TACO-compatible structure ready at: /workspace/processed_5cls/merged_data

Run manually:
  python preprocess.py \
      --taco_root   /workspace/processed_5cls/merged_data \
      --output_root /workspace/processed_5cls \
      --path all


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/data/merge_datasets.py', '--taco_root', '/workspace/TACO', '--external_root', '/workspace/external_datasets', '--output_root', '/workspace/processed_5cls', '--skip_preprocess'], returncode=0)

## 5) Pré-processamento Path A — normalmente NÃO rodar

A estrutura `train/val/test` já existe em `/workspace/processed_4cls`, então esta célula fica comentada por padrão.

Rode apenas se você apagou/recriou o dataset processado ou mudou o mapeamento de classes.


In [6]:
preprocess_script = DATA_DIR / 'preprocess.py'

run_cmd([
    sys.executable, str(preprocess_script),
    '--taco_root', str(PROCESSED_DIR / 'merged_data'),
    '--output_root', str(PROCESSED_DIR),
    '--path', 'A',
])


$ /workspace/.venv/bin/python /workspace/TrashScan/data/preprocess.py --taco_root /workspace/processed_5cls/merged_data --output_root /workspace/processed_5cls --path A


INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.8 (you have 1.4.10). Upgrade using: pip install --upgrade albumentations


loading annotations into memory...
Done (t=0.32s)
creating index...
index created!
  Detected pre-mapped 4-class annotations — skipping TACO remapping
Class counts before merge: {'glass': 234, 'other': 3438, 'plastic': 4249, 'paper': 671, 'metal': 687}
Class counts after merge: {'glass': 234, 'other': 3438, 'plastic': 4249, 'paper': 671, 'metal': 687}
Split → train: 6495  val: 1392  test: 1392
Class weights: {'plastic': 0.511278, 'metal': 1.125499, 'other': 0.488841, 'glass': 1.77672, 'paper': 1.097661}

[train] Processing 6495 images …


  Letterbox train: 100%|██████████| 6495/6495 [14:51<00:00,  7.28it/s]


  Running copy-paste oversampling …
  Oversampling cap: 4953  (2x median=1310)
    glass        have=500   generate=4453  final=4953
    metal        have=1246  generate=3707  final=4953
    other        have=6605  generate=0     final=6605
    paper        have=1310  generate=3643  final=4953
    plastic      have=6038  generate=0     final=6038
  [plastic] already at/above cap, skipping


...

  Letterbox test:   0%|          | 1/1392 [00:00<02:56,  7.88it/s]


[test] Processing 1392 images …


  Letterbox test: 100%|██████████| 1392/1392 [01:51<00:00, 12.51it/s]



Preprocessing complete.
Outputs written to: /workspace/processed_5cls
YOLO dataset config written: /workspace/processed_5cls/dataset_path_A.yaml


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/data/preprocess.py', '--taco_root', '/workspace/processed_5cls/merged_data', '--output_root', '/workspace/processed_5cls', '--path', 'A'], returncode=0)

## 6) Conferência do dataset Path A

Confirme que o YAML está apontando para o dataset correto e que usa as 4 classes esperadas.


In [7]:
import yaml

print('DATASET_YAML_PATH_A =', DATASET_YAML_PATH_A)
print('Existe?', DATASET_YAML_PATH_A.exists())

if DATASET_YAML_PATH_A.exists():
    with open(DATASET_YAML_PATH_A, 'r') as f:
        data_yaml = yaml.safe_load(f)
    print(data_yaml)


DATASET_YAML_PATH_A = /workspace/processed_5cls/dataset_path_A.yaml
Existe? True
{'path': '/workspace/processed_5cls', 'train': 'train/path_A/images', 'val': 'val/path_A/images', 'test': 'test/path_A/images', 'nc': 5, 'names': ['plastic', 'paper', 'metal', 'glass', 'other']}


## 7) Limpeza opcional de cache `.npy` criado por `cache='disk'`

Se o volume cresceu muito depois de treinar, provavelmente foram criados arquivos `.npy` dentro de `processed_4cls`. Esta célula apenas mostra o tamanho; a deleção fica comentada.

Para evitar recriar isso, no `train_path_A.py` use `cache=False` ou `cache=True`, mas não `cache='disk'`.


In [8]:
run_cmd('du -sh /workspace/processed_5cls || true')
run_cmd("find /workspace/processed_5cls -type f -name '*.npy' | wc -l")


# run_cmd("find /workspace/processed_5cls -type f -name '*.npy' -delete")
# run_cmd('du -sh /workspace/processed_5cls || true')

$ du -sh /workspace/processed_5cls || true
14G	/workspace/processed_5cls
$ find /workspace/processed_5cls -type f -name '*.npy' | wc -l
0


CompletedProcess(args="find /workspace/processed_5cls -type f -name '*.npy' | wc -l", returncode=0)

## 8) Treino Path A

Esta célula roda `train_path_A.py` com saída em `/workspace/runs/path_A`.

Importante: os pesos de **`yolov11m`**, **`yolov8m`** e **`yolov9s`** já foram calculados e estão em `runs`. Rode novamente apenas se quiser retreinar.


In [10]:
train_script = TRAIN_DIR / 'train_path_A.py'

run_cmd([
    sys.executable, str(train_script),
    '--data', str(DATASET_YAML_PATH_A),
    '--output', str(RUNS_PATH_A_DIR),
    '--models', *MODELS,
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--imgsz', str(IMGSZ),
    '--device', str(DEVICE),
    '--patience', str(PATIENCE),
    '--mlflow_uri', str(MLFLOW_DIR),
])


$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_A.py --data /workspace/processed_5cls/dataset_path_A.yaml --output /workspace/runs/path_A_5cls --models yolov11m --epochs 300 --batch 16 --imgsz 640 --device 0 --patience 15 --mlflow_uri /root/mlflow
GPU  : NVIDIA RTX A4500
VRAM : 21.0 GB
Loaded class weights: {'plastic': 0.511, 'paper': 1.098, 'metal': 1.125, 'glass': 1.777, 'other': 0.489}

────────────────────────────────────────────────────────────
  Model  : yolov11m  (yolo11m.pt)
  Data   : /workspace/processed_5cls/dataset_path_A.yaml
  Epochs : 300   Batch : 16   imgsz : 640
────────────────────────────────────────────────────────────
New https://pypi.org/project/ultralytics/8.4.49 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.2 🚀 Python-3.11.10 torch-2.3.1+cu121 CUDA:0 (NVIDIA RTX A4500, 20053MiB)
engine/trainer: task=detect, mode=train, model=yolo11m.pt, data=/workspace/processed_5cls/dataset_path_A.yaml, epochs=300, time=None, pa

train: Scanning /workspace/processed_5cls/train/path_A/labels.cache... 18294 images, 4 backgrounds, 0 corrupt: 100%|██████████| 18294/18294 [00:00<?, ?it/s]


WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


train: Caching images (20.9GB RAM): 100%|██████████| 18294/18294 [00:36<00:00, 499.80it/s]
INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.8 (you have 1.4.10). Upgrade using: pip install --upgrade albumentations


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01), CLAHE(p=0.01, clip_limit=(1, 4.0), tile_grid_size=(8, 8))


...

    109/300      9.44G     0.5498     0.6852      1.062         11        640: 100%|██████████| 1144/1144 [04:21<00:00,  4.38it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 44/44 [00:09<00:00,  4.76it/s]


                   all       1392       3446      0.664      0.487      0.552      0.382
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 94, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

109 epochs completed in 8.267 hours.
Optimizer stripped from /workspace/runs/path_A_5cls/yolov11m/weights/last.pt, 40.6MB
Optimizer stripped from /workspace/runs/path_A_5cls/yolov11m/weights/best.pt, 40.6MB

Validating /workspace/runs/path_A_5cls/yolov11m/weights/best.pt...
Ultralytics 8.3.2 🚀 Python-3.11.10 torch-2.3.1+cu121 CUDA:0 (NVIDIA RTX A4500, 20053MiB)
YOLO11m summary (fused): 303 layers, 20,033,887 parameters, 0 gradients, 67.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 44/44 [00:17<00:00,  2.49it/s]


                   all       1392       3446      0.686       0.51      0.574      0.386
               plastic        767       1411      0.668      0.533      0.584      0.365
                 paper        202        266      0.623      0.497      0.532      0.353
                 metal        182        282      0.724      0.571      0.671      0.451
                 glass         54        107      0.702      0.364      0.429      0.262
                 other        622       1380       0.71      0.585      0.653      0.497
Speed: 0.1ms preprocess, 9.8ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /workspace/runs/path_A_5cls/yolov11m
MLflow: results logged to runs/mlflow
MLflow: disable with 'yolo settings mlflow=False'
  Latency: 33.45 ms/image

  [yolov11m] mAP50=0.5739  mAP50-95=0.3858  latency=33.45ms

PATH A  —  YOLO BENCHMARK SUMMARY
          mAP50  mAP50_95  precision  recall  box_loss  cls_loss  latency_ms
model                                      

CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_A.py', '--data', '/workspace/processed_5cls/dataset_path_A.yaml', '--output', '/workspace/runs/path_A_5cls', '--models', 'yolov11m', '--epochs', '300', '--batch', '16', '--imgsz', '640', '--device', '0', '--patience', '15', '--mlflow_uri', '/root/mlflow'], returncode=0)

## 9) Conferir pesos disponíveis

O avaliador espera encontrar os modelos no formato:

```text
/root/runs/path_A/<modelo>/weights/best.pt
```


In [ ]:
best_weights = sorted(RUNS_PATH_A_DIR.glob('*/weights/best.pt'))
print(f'Pesos encontrados em {RUNS_PATH_A_DIR}: {len(best_weights)}')
for p in best_weights:
    print(' -', p)

if not best_weights:
    raise FileNotFoundError(f'Nenhum best.pt encontrado em {RUNS_PATH_A_DIR}')


## 10) Resumo rápido dos treinos

In [ ]:
run_cmd([
    sys.executable, str(TRAIN_DIR / 'train_path_A.py'),
    '--summarize',
    '--output', str(RUNS_PATH_A_DIR),
])


## 11) Avaliação Path A

In [12]:
evaluate_script = EVAL_DIR / 'evaluate.py'

run_cmd([
    sys.executable, str(evaluate_script),
    '--path', 'A',
    '--runs_dir', str(RUNS_PATH_A_DIR),
    '--data_yaml', str(DATASET_YAML_PATH_A),
    '--output', str(RESULTS_PATH_A_DIR),
    '--device', str(DEVICE),
    '--imgsz', str(IMGSZ),
])


$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate.py --path A --runs_dir /workspace/runs/path_A_5cls --data_yaml /workspace/processed_5cls/dataset_path_A.yaml --output /workspace/results_path_A_5cls --device 0 --imgsz 640

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov11m
  Weights   : /workspace/runs/path_A_5cls/yolov11m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [01:29<00:00, 15.63it/s]



  mAP@0.5    : 0.5699
  mAP@0.5:95 : 0.3381
  Precision  : 0.5613
  Recall     : 0.4599
  F1         : 0.5015
  Latency    : 10.53 ms  (95.0 FPS)
  Params     : 20.1M
  Size       : 40.6 MB
  Saved      : /workspace/results_path_A_5cls/individual/A_yolov11m_yolov11m.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov8m
  Weights   : /workspace/runs/path_A_5cls/yolov8m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference:  60%|██████    | 837/1392 [00:39<00:26, 21.33it/s]
Traceback (most recent call last):
  File "/workspace/TrashScan/eval/evaluate.py", line 1325, in <module>
    evaluate_model(
  File "/workspace/TrashScan/eval/evaluate.py", line 765, in evaluate_model
    pred_boxes, pred_scores, pred_classes, gt_boxes, gt_classes = collect_predictions(
                                                                  ^^^^^^^^^^^^^^^^^^^^
  File "/workspace/TrashScan/eval/evaluate.py", line 364, in collect_predictions
    r = model.predict(
        ^^^^^^^^^^^^^^
  File "/workspace/.venv/lib/python3.11/site-packages/ultralytics/engine/model.py", line 554, in predict
    return self.predictor.predict_cli(source=source) if is_cli else self.predictor(source=source, stream=stream)
                                                                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/.venv/lib/python3.11/site-packages/ultralytics/engine/predictor.py", line 168, in _

KeyboardInterrupt: 

## 12) Avaliação TTA + WBF — opcional

A célula abaixo fica comentada porque é mais lenta. Rode apenas se quiser avaliar com test-time augmentation e weighted boxes fusion.


In [ ]:
run_cmd([
    sys.executable, str(EVAL_DIR / 'evaluate.py'),
    '--path', 'A',
    '--runs_dir', str(RUNS_PATH_A_DIR),
    '--data_yaml', str(DATASET_YAML_PATH_A),
    '--output', str(RESULTS_PATH_A_DIR),
    '--device', str(DEVICE),
    '--imgsz', str(IMGSZ),
    "--use_tta_wbf",
    "--tta_scales", "512", "640", "768",
    "--tta_flip",
    "--tta_wbf_iou", "0.55",
    "--tta_skip_box_thr", "0.001",
])


## 13) Grid Search TTA + WBF

In [ ]:
import itertools
import subprocess
import sys

# Defina os ranges (ajuste conforme necessário)
tta_wbf_iou_values = [0.45, 0.50, 0.55, 0.60, 0.65]
tta_skip_box_thr_values = [0.0001, 0.001, 0.01, 0.05]

results = []

for iou, skip_thr in itertools.product(tta_wbf_iou_values, tta_skip_box_thr_values):
    print(f"Rodando com IOU={iou}, SKIP_THR={skip_thr}")

    cmd = [
        sys.executable, str(EVAL_DIR / 'evaluate_tta.py'),
        '--path', 'A',
        '--runs_dir', str(RUNS_PATH_A_DIR),
        '--data_yaml', str(DATASET_YAML_PATH_A),
        '--output', str(RESULTS_PATH_A_DIR / f"iou_{iou}_skip_{skip_thr}"),
        '--device', str(DEVICE),
        '--imgsz', str(IMGSZ),
        "--use_tta_wbf",
        "--tta_scales", "512", "640", "768",
        "--tta_flip",
        "--tta_wbf_iou", str(iou),
        "--tta_skip_box_thr", str(skip_thr),
    ]

    subprocess.run(cmd)

    # opcional: guardar configs testadas
    results.append({
        "iou": iou,
        "skip_thr": skip_thr,
        "output_dir": str(RESULTS_PATH_A_DIR / f"iou_{iou}_skip_{skip_thr}")
    })

print("Grid search finalizado!")

## 14) Checklist final

Antes de parar/deletar o pod, confirme que o que você quer manter está no volume:

- pesos essenciais em `/workspace/runs/path_A`
- resultados em `/workspace/results_path_A`
- projeto em `/workspace/TrashScan`
- dataset em `/workspace/processed_4cls`


In [ ]:
run_cmd('du -sh /workspace/runs /workspace/results_path_A /workspace/processed_4cls /workspace/TrashScan 2>/dev/null || true')
